## Parte 1 – Scraping de decretos de emergencia [5 pts]


In [1]:
import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

meses = [
    ["01-01-2025", "31-01-2025"],
    ["01-02-2025", "28-02-2025"],
    ["01-03-2025", "31-03-2025"],
    ["01-04-2025", "30-04-2025"],
    ["01-05-2025", "31-05-2025"],
]

url_base = "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent"

In [2]:
respuesta = requests.get("https://www.gob.pe/robots.txt", headers={"User-Agent": "Mozilla/5.0"})
print(respuesta.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



### 1. `Robots.txt`

**¿Qué prohíbe?**

Para cualquier bot (`User-agent: *`) solo prohíbe tres tipos de ruta:

- `Disallow: /admin/`: la zona de administración del sitio.
- `Disallow: /*?sheet=` y `Disallow: /*&sheet=`: cualquier URL que lleve el parámetro `sheet`.

Además, a algunos bots les pone reglas propias:

- **AhrefsBot**: `Disallow: /`, es decir, tiene prohibido todo el sitio.
- **GPTBot**: las mismas prohibiciones que el resto, más `Crawl-delay: 5`.
- **OAI-SearchBot**: las mismas prohibiciones, más `Crawl-delay: 2`.

**¿Está permitida la página de búsquedas (`/busquedas`)?**

Sí. `/busquedas` no empieza por `/admin/`, y las URLs que usamos (`/busquedas?contenido[]=normas&institucion[]=pcm&term=...&desde=...&hasta=...`) no tienen el parámetro `sheet`. Ninguna regla de `User-agent: *` las alcanza, y en `robots.txt` todo lo que no está prohibido está permitido. Las páginas de cada norma (`/institucion/pcm/normas-legales/...`) también están permitidas.

**¿Por qué igual conviene hacer pausas entre página y página?**

Que algo esté permitido no quiere decir que se pueda pedir a cualquier velocidad. El `Crawl-delay` es el número de segundos que el sitio pide esperar entre una petición y la siguiente. Aunque a nuestro bot no se le pide de forma explícita, que gob.pe se lo pida a GPTBot (5 s) y a OAI-SearchBot (2 s) muestra lo que el servidor considera un ritmo razonable. Además, si pedimos páginas muy seguidas, el sitio puede tomarnos por un ataque y bloquearnos la IP o devolver páginas incompletas.

Por eso el código espera `time.sleep(3)` después de cargar cada mes en Selenium y `time.sleep(1)` entre las páginas de cada norma (como pide la tarea).

### 2. Pedimos la busqueda para todo el mes de Enero.

In [3]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenta para ocultarlo
opciones.add_argument("--window-size=1400,900")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2025&hasta=31-01-2025")

WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/normas-legales/']"))
)
time.sleep(3)

print("Título:", navegador.title)


Título: Plataforma del Estado Peruano


### 4. Titulo, fecha, descripcion y enlace

In [4]:
tarjetas = navegador.find_elements(By.CSS_SELECTOR, "li[role='listitem'] article")

normas_enero = []

for tarjeta in tarjetas:
    enlace = tarjeta.find_element(By.CSS_SELECTOR, "a.link-principal")
    fecha = tarjeta.find_element(By.CSS_SELECTOR, "span.text-gray-900.font-bold")
    descripcion = tarjeta.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    norma = {
        "titulo": enlace.text,
        "fecha": fecha.text.replace("Publicado: ", ""),
        "descripcion": descripcion.text,
        "enlace": enlace.get_attribute("href"),
    }
    normas_enero.append(norma)

df_enero = pd.DataFrame(normas_enero)
df_enero

,titulo,fecha,descripcion,enlace
0,Decreto Supremo N.° 018-2025-PCM,30 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 016-2025-PCM,29 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 013-2025-PCM,23 de enero de 2025,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 012-2025-PCM,22 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 011-2025-PCM,22 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 008-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 006-2025-PCM,8 de enero de 2025,Decreto Supremo que prorroga el Estado de Eme...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Decreto Supremo N.° 004-2025-PCM,3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 003-2025-PCM,3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


### 5. Repetimos para todos los meses

In [5]:
normas = []
verificacion = []

for desde, hasta in meses:
    navegador.get(url_base + "&desde=" + desde + "&hasta=" + hasta)

    WebDriverWait(navegador, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/normas-legales/']"))
    )
    time.sleep(3)

    # Número total de resultados que muestra la página ("12 Resultados")
    texto_resultados = navegador.find_element(
        By.CSS_SELECTOR, "h2[role='status'] span"
    ).text
    resultados_totales = int(texto_resultados.split(" ")[0])

    tarjetas = navegador.find_elements(By.CSS_SELECTOR, "li[role='listitem'] article")

    for tarjeta in tarjetas:
        enlace = tarjeta.find_element(By.CSS_SELECTOR, "a.link-principal")
        fecha = tarjeta.find_element(By.CSS_SELECTOR, "span.text-gray-900.font-bold")
        descripcion = tarjeta.find_element(By.CSS_SELECTOR, "p[id$='-description']")

        norma = {
            "titulo": enlace.text,
            "fecha": fecha.text.replace("Publicado: ", ""),
            "descripcion": descripcion.text.strip(),
            "enlace": enlace.get_attribute("href"),
        }
        normas.append(norma)

    # Guardamos los dos números de este mes para la verificación 
    verificacion.append({
        "mes": desde[3:],  # "01-01-2025" → "01-2025"
        "resultados_totales": resultados_totales,
        "resultados_extraidos": len(tarjetas),
    })

    print(desde, "a", hasta, "→", len(tarjetas), "normas")

df_normas = pd.DataFrame(normas)

01-01-2025 a 31-01-2025 → 12 normas
01-02-2025 a 28-02-2025 → 4 normas
01-03-2025 a 31-03-2025 → 12 normas
01-04-2025 a 30-04-2025 → 15 normas
01-05-2025 a 31-05-2025 → 11 normas


### 6. Verificamos

In [6]:
df_verificacion = pd.DataFrame(verificacion)

# Comprobamos que los dos números coincidan en todos los meses
df_verificacion["coinciden"] = (
    df_verificacion["resultados_totales"] == df_verificacion["resultados_extraidos"]
)

if df_verificacion["coinciden"].all():
    print("Los dos números coinciden en todos los meses. Podemos seguir.")
else:
    meses_mal = df_verificacion.loc[~df_verificacion["coinciden"], "mes"].tolist()
    print("No coinciden en:", meses_mal, "→ revisar el código antes de seguir.")

df_verificacion

Los dos números coinciden en todos los meses. Podemos seguir.


,mes,resultados_totales,resultados_extraidos,coinciden
0,01-2025,12,12,True
1,02-2025,4,4,True
2,03-2025,12,12,True
3,04-2025,15,15,True
4,05-2025,11,11,True


### 7. Eliminamos las normas repetidas

In [7]:
normas_unicas = []
enlaces_vistos = []

for norma in normas:
    if norma["enlace"] not in enlaces_vistos:
        normas_unicas.append(norma)
        enlaces_vistos.append(norma["enlace"])

cantidad_repetidas = len(normas) - len(normas_unicas)

print("Normas antes de quitar repetidas:", len(normas))
print("Normas únicas:", len(normas_unicas))
print("Normas repetidas quitadas:", cantidad_repetidas)

# A partir de aquí seguimos trabajando solo con las normas únicas
normas = normas_unicas

Normas antes de quitar repetidas: 54
Normas únicas: 54
Normas repetidas quitadas: 0


### 8. Filtramos normas de la PCM

In [8]:
normas_pcm = []

for norma in normas:
    if "/institucion/pcm/" in norma["enlace"]:
        normas_pcm.append(norma)

cantidad_quitadas = len(normas) - len(normas_pcm)

print("Normas antes del filtro:", len(normas))
print("Normas de la PCM:", len(normas_pcm))
print("Normas quitadas:", cantidad_quitadas)


Normas antes del filtro: 54
Normas de la PCM: 53
Normas quitadas: 1


### 9. Sacamos el título completo de cada norma

In [9]:
for norma in normas_pcm:
    # gob.pe rechaza los pedidos que no se identifican como navegador
    respuesta = requests.get(norma["enlace"], headers={"User-Agent": "Mozilla/5.0"})
    sopa = BeautifulSoup(respuesta.text, "html.parser")

    descripcion = sopa.find("div", class_="description")

    if descripcion is None:
        # Si la página no carga bien, nos quedamos con el título cortado de la lista
        print("No se encontró el título en:", norma["enlace"])
        norma["titulo_completo"] = norma["descripcion"]
    else:
        # El primer <div> de adentro tiene solo el título,
        # sin el texto extra "DS N° ... PDF ... Descargar"
        norma["titulo_completo"] = descripcion.find("div").text.strip()

    # Pausa de 1 segundo entre páginas
    time.sleep(1)

df_normas_pcm = pd.DataFrame(normas_pcm)

cortados = df_normas_pcm["descripcion"].str.endswith("...").sum()
print("Normas con el título cortado en la lista:", cortados)

df_normas_pcm[["titulo", "descripcion", "titulo_completo"]]

Normas con el título cortado en la lista: 24


,titulo,descripcion,titulo_completo
0,Decreto Supremo N.° 018-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
1,Decreto Supremo N.° 016-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
2,Decreto Supremo N.° 013-2025-PCM,Decreto Supremo que modifica el Decreto Suprem...,Decreto Supremo que modifica el Decreto Suprem...
3,Decreto Supremo N.° 012-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
4,Decreto Supremo N.° 011-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
5,Decreto Supremo N.° 008-2025-PCM,Declaratoria del Estado de Emergencia,Declaratoria del Estado de Emergencia
6,Decreto Supremo N.° 007-2025-PCM,Declaratoria del Estado de Emergencia,Declaratoria del Estado de Emergencia
7,Decreto Supremo N.° 006-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
8,Decreto Supremo N.° 004-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...
9,Decreto Supremo N.° 003-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,Decreto Supremo que prorroga el Estado de Emer...


### 10. Creamos las columnas `es_lluvia`, `tipo` y `motivo`

In [10]:
for norma in normas_pcm:
    titulo = norma["titulo_completo"].lower()

    # es_lluvia: True si habla de lluvias o precipitaciones
    norma["es_lluvia"] = "lluvia" in titulo or "precipitaciones" in titulo

    # tipo: primero vemos si es prórroga; si no, si es declaratoria
    if "prórroga" in titulo or "prorroga" in titulo:
        norma["tipo"] = "prorroga"
    elif "declara" in titulo:
        norma["tipo"] = "declara"
    else:
        norma["tipo"] = "otro"

    # motivo: peligro inminente o impacto de daños
    if "peligro inminente" in titulo:
        norma["motivo"] = "peligro inminente"
    elif "impacto de daños" in titulo:
        norma["motivo"] = "impacto de daños"
    else:
        norma["motivo"] = "otro"

df_normas_pcm = pd.DataFrame(normas_pcm)

print(df_normas_pcm["tipo"].value_counts(), "\n")
print(df_normas_pcm["motivo"].value_counts(), "\n")

# Revisamos las filas que quedaron como "otro"
otros = df_normas_pcm[(df_normas_pcm["tipo"] == "otro") | (df_normas_pcm["motivo"] == "otro")]
print("Filas con 'otro' en tipo o motivo:", len(otros))
print("De esas, ¿cuántas son de lluvias?:", otros["es_lluvia"].sum())

otros[["titulo", "titulo_completo", "es_lluvia", "tipo", "motivo"]]

tipo
prorroga    37
declara     16
Name: count, dtype: int64 

motivo
otro                 33
impacto de daños     17
peligro inminente     3
Name: count, dtype: int64 

Filas con 'otro' en tipo o motivo: 33
De esas, ¿cuántas son de lluvias?: 0


,titulo,titulo_completo,es_lluvia,tipo,motivo
0,Decreto Supremo N.° 018-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
1,Decreto Supremo N.° 016-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
2,Decreto Supremo N.° 013-2025-PCM,Decreto Supremo que modifica el Decreto Suprem...,False,prorroga,otro
3,Decreto Supremo N.° 012-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
4,Decreto Supremo N.° 011-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
5,Decreto Supremo N.° 008-2025-PCM,Declaratoria del Estado de Emergencia,False,declara,otro
6,Decreto Supremo N.° 007-2025-PCM,Declaratoria del Estado de Emergencia,False,declara,otro
8,Decreto Supremo N.° 004-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
9,Decreto Supremo N.° 003-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro
10,Decreto Supremo N.° 002-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...,False,prorroga,otro


#### Revisión de las filas "otro"

**`tipo = "otro"`: no quedó ninguna.** Las 53 normas dicen "prorroga" o "declara" en el título. Hay que tener en cuenta dos casos:

- Los DS 007-2025-PCM y 008-2025-PCM tienen como título completo solo *"Declaratoria del Estado de Emergencia"*. Quedan como `declara` porque "declaratoria" contiene "declara", y eso está bien. Pero como el título no dice dónde ni por qué, quedan como `es_lluvia = False` y `motivo = "otro"`. Para saber si son de lluvias habría que abrir el PDF.
- El DS 013-2025-PCM *"modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga…"* quedó como `prorroga` porque contiene esa palabra. En realidad es una **modificación** de una prórroga. No afecta el análisis porque no es de lluvias.

**`motivo = "otro"`: son normas que no son de lluvias.** Ninguna norma de lluvias quedó como "otro": todas dicen "impacto de daños" o "peligro inminente". Las que quedaron como "otro" son emergencias de otro tipo (Lima Metropolitana y Callao, Trujillo, Pataz, Corredor Vial Sur, etc.) cuyo título no menciona ninguna de las dos frases. Están bien clasificadas, y además se eliminan en el paso siguiente, cuando nos quedamos solo con las de lluvias.

### 11. Nos quedamos solo con las normas de lluvias

In [11]:
normas_lluvias = []

for norma in normas_pcm:
    if norma["es_lluvia"]:
        normas_lluvias.append(norma)

df_lluvias = pd.DataFrame(normas_lluvias)

print("Normas de lluvias:", len(normas_lluvias))

df_lluvias[["titulo", "fecha", "titulo_completo", "tipo", "motivo"]].head(5)

Normas de lluvias: 17


,titulo,fecha,titulo_completo,tipo,motivo
0,Decreto Supremo N.° 024-2025-PCM,26 de febrero de 2025,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
1,Decreto Supremo N.° 021-2025-PCM,19 de febrero de 2025,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
2,Decreto Supremo N.° 034-2025-PCM,12 de marzo de 2025,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
3,Decreto Supremo N.° 033-2025-PCM,12 de marzo de 2025,Decreto Supremo que declara el Estado de Emerg...,declara,peligro inminente
4,Decreto Supremo N.° 026-2025-PCM,5 de marzo de 2025,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños


### 12. Departamentos que menciona cada decreto

In [12]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]

for norma in normas_lluvias:
    departamentos_norma = []

    # Recorremos cada departamento UNA sola vez: aunque aparezca dos veces
    # en el título, solo se agrega una vez a la lista de este decreto
    for departamento in departamentos:
        # Buscamos en el título original (con mayúsculas) y por palabra completa
        if re.search(r"\b" + departamento + r"\b", norma["titulo_completo"]):
            departamentos_norma.append(departamento)

    norma["departamentos"] = departamentos_norma

df_lluvias = pd.DataFrame(normas_lluvias)
df_lluvias[["titulo", "tipo", "departamentos"]]

,titulo,tipo,departamentos
0,Decreto Supremo N.° 024-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
1,Decreto Supremo N.° 021-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
2,Decreto Supremo N.° 034-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
3,Decreto Supremo N.° 033-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
4,Decreto Supremo N.° 026-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
5,Decreto Supremo N.° 057-2025-PCM,declara,[Loreto]
6,Decreto Supremo N.° 056-2025-PCM,prorroga,"[Áncash, Cusco, Huánuco, La Libertad, Pasco, P..."
7,Decreto Supremo N.° 053-2025-PCM,prorroga,"[Amazonas, Áncash, Cusco, Piura, Tumbes, Ucayali]"
8,Decreto Supremo N.° 052-2025-PCM,declara,"[Cusco, Madre de Dios]"
9,Decreto Supremo N.° 038-2025-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."


In [13]:
# Comprobación con un ejemplo de nuestra temporada: DS 038-2025-PCM
for norma in normas_lluvias:
    if "038-2025" in norma["titulo"]:
        ejemplo = norma["titulo_completo"]

print(ejemplo, "\n")

# Forma ingenua: buscar "ica" en el título en minúsculas
print("¿'ica' en minúsculas?:", "ica" in ejemplo.lower())

# Forma correcta: título original y palabra completa
print("¿'Ica' como palabra completa?:", re.search(r"\bIca\b", ejemplo) is not None)

Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Junín, Lima, Loreto, Madre de Dios, Pasco, Puno, San Martín y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales 

¿'ica' en minúsculas?: True
¿'Ica' como palabra completa?: False


### ¿Cómo evitamos contar mal?

**Problema 1: "ica" dentro de "Huancavel*ica*".** En nuestra temporada, el **DS 038-2025-PCM** (21 de abril de 2025) declara emergencia en *"…Cajamarca, Cusco, **Huancavelica**, Huánuco, Junín, Lima…"* y **no menciona Ica**. Si buscáramos `"ica"` en el título en minúsculas, este decreto contaría también para Ica, porque "huancavel**ica**" contiene esas letras. Lo mismo pasa con el **DS 043-2025-PCM**. La celda anterior lo comprueba: la búsqueda ingenua da `True` y la búsqueda por palabra completa da `False`.

Para evitarlo buscamos en el **título original, con mayúsculas**, y por **palabra completa** con `re.search(r"\b" + departamento + r"\b", titulo)`. El `\b` marca el borde de una palabra, así que `\bIca\b` solo encuentra "Ica" suelta y no dentro de "Huancavelica".

**Problema 2: un mismo departamento nombrado dos veces** (por ejemplo, *"provincia de Ica del departamento de Ica"*). El bucle recorre la lista de 25 departamentos **una sola vez por decreto** y solo pregunta si cada departamento aparece o no. Aunque "Ica" aparezca dos veces en el título, se agrega una sola vez a la lista de ese decreto, y por eso cuenta una sola vez. En nuestra temporada los decretos de lluvias nombran los departamentos en una sola lista (*"…de los departamentos de Amazonas, Áncash, …"*), pero el código igual está protegido para ese caso.

### Guardamos los archivos en la carpeta `datos/`

In [14]:
os.makedirs("datos", exist_ok=True)

# 1) decretos_lluvias.csv
# En nuestras listas, "titulo" guarda el número de la norma (texto del enlace)
# y "titulo_completo" el título largo, así que los renombramos como pide la tarea
decretos_lluvias = df_lluvias.rename(columns={
    "titulo": "numero",
    "titulo_completo": "titulo",
})
decretos_lluvias = decretos_lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]]
decretos_lluvias.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")

# 2) decretos_por_departamento.csv
filas = []
for departamento in departamentos:
    declaratorias = 0
    prorrogas = 0
    for norma in normas_lluvias:
        if departamento in norma["departamentos"]:
            if norma["tipo"] == "declara":
                declaratorias = declaratorias + 1
            elif norma["tipo"] == "prorroga":
                prorrogas = prorrogas + 1

    filas.append({
        "departamento": departamento,
        "declaratorias": declaratorias,
        "prorrogas": prorrogas,
    })

decretos_por_departamento = pd.DataFrame(filas)
decretos_por_departamento.to_csv(
    "datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig"
)

print("Guardado datos/decretos_lluvias.csv:", len(decretos_lluvias), "filas")
print("Guardado datos/decretos_por_departamento.csv:", len(decretos_por_departamento), "filas")

decretos_por_departamento

Guardado datos/decretos_lluvias.csv: 17 filas
Guardado datos/decretos_por_departamento.csv: 25 filas


,departamento,declaratorias,prorrogas
0,Amazonas,7,4
1,Áncash,7,6
2,Apurímac,6,2
3,Arequipa,7,2
4,Ayacucho,7,4
5,Cajamarca,6,2
6,Callao,1,0
7,Cusco,8,6
8,Huancavelica,7,2
9,Huánuco,7,5
